# Road Accident Severity – Temporal Generalisation Study (2020-2022)

**Authors:** Amru Alyan, Nagham Mhesen, Abdelrahman Muhtaseb  
**Course:** למידה חישובית (Computational Learning)  
**Institution:** [Azrieli College of Engineering Jerusalem]  
**Date:** July 2025

## Project Overview

This project aims to predict road accident severity using temporal generalization, where we train on 2020-2021 data and test on 2022 data. We focus on predicting injury severity levels (Fatal, Severe, Minor) using various machine learning algorithms including Decision Trees, Random Forest, and AdaBoost.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Machine Learning imports
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Set style for plots
plt.style.use('default')
sns.set_palette("husl")

# Helper function to load CSV files and add year column
def load_accident_data(file_paths, year_col=True):
    """
    Load accident data from multiple CSV files and add a YEAR column.
    
    Args:
        file_paths (list): List of CSV file paths
        year_col (bool): Whether to add a YEAR column based on filename
    
    Returns:
        pd.DataFrame: Combined dataframe with accident data
    """
    dataframes = []
    
    for file_path in file_paths:
        print(f"Loading {file_path}...")
        df = pd.read_csv(file_path)
        
        if year_col:
            # Extract year from filename
            year = int(file_path.split('_')[-1].split('.')[0])
            df['YEAR'] = year
            
        dataframes.append(df)
        print(f"  Shape: {df.shape}")
    
    # Combine all dataframes
    combined_df = pd.concat(dataframes, ignore_index=True)
    print(f"\nCombined dataset shape: {combined_df.shape}")
    
    return combined_df

print("Libraries imported successfully!")

## 1. Environment Setup and Data Dictionary Integration

This section loads the required libraries and sets up the data dictionary integration for decoding categorical variables in the Israeli traffic accident dataset.

### Key Components:
- **Import Libraries:** Essential data science libraries (pandas, numpy, sklearn, matplotlib, seaborn)
- **Data Dictionary Loading:** Functions to load and parse Hebrew categorical mappings from Excel files
- **Data Decoding Functions:** Utilities to apply mappings and create meaningful categorical labels

In [ ]:
# =============================================================================
# 2. Load Datasets 
# =============================================================================

# Load training data (2020-2021)
train_files = ["data/accidents_2020.csv", "data/accidents_2021.csv"]
test_file = "data/accidents_2022.csv"

# Load training data
print("Loading training data...")
train_data = load_accident_data(train_files)
print(f"Training data loaded: {train_data.shape}")

# Load test data
print("Loading test data...")
test_data = load_accident_data([test_file])
print(f"Test data loaded: {test_data.shape}")

print("\n=== Data Overview ===")
print(f"Training data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")
print(f"Total samples: {train_data.shape[0] + test_data.shape[0]}")

# Basic data info
print(f"\nTraining data columns: {train_data.shape[1]}")
print(f"Test data columns: {test_data.shape[1]}")

# Check basic statistics
print(f"\nTraining data missing values: {train_data.isnull().sum().sum()}")
print(f"Test data missing values: {test_data.isnull().sum().sum()}")

In [ ]:
# Load data dictionaries and create mapping functions
import openpyxl

def load_data_mappings():
    """
    Load data mappings from the Excel dictionary files
    """
    mappings = {}
    
    try:
        # Load dictionary22.xlsx
        workbook = openpyxl.load_workbook('data/dictionary22.xlsx')
        
        # Iterate through all sheets
        for sheet_name in workbook.sheetnames:
            sheet = workbook[sheet_name]
            
            # Skip empty sheets
            if sheet.max_row <= 1:
                continue
                
            # Read the mapping data (assuming first column is code, second is description)
            sheet_mappings = {}
            for row in range(2, sheet.max_row + 1):  # Skip header row
                code_cell = sheet.cell(row=row, column=1).value
                desc_cell = sheet.cell(row=row, column=2).value
                
                if code_cell is not None and desc_cell is not None:
                    # Convert to appropriate type
                    try:
                        code = int(code_cell) if str(code_cell).isdigit() else code_cell
                    except:
                        code = code_cell
                    
                    # Clean description
                    description = str(desc_cell).strip()
                    sheet_mappings[code] = description
            
            if sheet_mappings:
                mappings[sheet_name] = sheet_mappings
                
        workbook.close()
        
    except Exception as e:
        print(f"Error loading dictionary22.xlsx: {e}")
    
    return mappings

def apply_data_mappings(df, mappings):
    """
    Apply data mappings to decode categorical variables
    """
    df_decoded = df.copy()
    
    # Common column mappings based on typical accident data structure
    column_mapping_keys = {
        'MEZEG_AVIR': 'weather',
        'SUG_DEREH': 'road_type', 
        'MAHOZ': 'district',
        'YOM_LAYLA': 'day_night',
        'REHEV_1': 'vehicle_type_1',
        'REHEV_2': 'vehicle_type_2',
        'SHLITA_REHEV_1': 'vehicle_control_1',
        'SHLITA_REHEV_2': 'vehicle_control_2',
        'GIL_NEHAG_1': 'driver_age_1',
        'GIL_NEHAG_2': 'driver_age_2',
        'MIN_NEHAG_1': 'driver_gender_1',
        'MIN_NEHAG_2': 'driver_gender_2',
        'SIBAT_TEUNA': 'accident_cause',
        'SUG_TEUNA': 'accident_type',
        'HUMRAT_TEUNA': 'accident_severity'
    }
    
    # Apply mappings where available
    for col in df.columns:
        if col in column_mapping_keys:
            mapping_key = column_mapping_keys[col]
            
            # Try to find corresponding mapping in the loaded dictionaries
            for dict_key, mapping_dict in mappings.items():
                if mapping_key.lower() in dict_key.lower() or col in dict_key:
                    try:
                        df_decoded[f'{col}_decoded'] = df[col].map(mapping_dict)
                        print(f"Applied mapping for {col} -> {col}_decoded")
                        break
                    except Exception as e:
                        print(f"Failed to apply mapping for {col}: {e}")
    
    return df_decoded

def create_meaningful_features(df):
    """
    Create meaningful features based on decoded data
    """
    df_enhanced = df.copy()
    
    # Weather risk scoring
    weather_risk_map = {
        'clear': 1, 'sunny': 1, 'fine': 1,
        'cloudy': 2, 'overcast': 2,
        'light_rain': 3, 'drizzle': 3,
        'rain': 4, 'heavy_rain': 5,
        'fog': 5, 'mist': 4,
        'snow': 5, 'storm': 5
    }
    
    # Time-based features
    if 'SHAA' in df.columns:
        df_enhanced['hour_category'] = pd.cut(df['SHAA'], 
                                            bins=[0, 6, 12, 18, 24], 
                                            labels=['night', 'morning', 'afternoon', 'evening'],
                                            include_lowest=True)
        
        # Rush hour indicator
        df_enhanced['is_rush_hour'] = ((df['SHAA'] >= 7) & (df['SHAA'] <= 9)) | \
                                     ((df['SHAA'] >= 17) & (df['SHAA'] <= 19))
    
    # Weekend indicator
    if 'YOM_HASHAVUA' in df.columns:
        df_enhanced['is_weekend'] = df['YOM_HASHAVUA'].isin([6, 7])  # Assuming 6,7 are weekend
    
    return df_enhanced

# Load and apply mappings
print("=== Loading Data Dictionaries ===")
data_mappings = load_data_mappings()
print(f"Loaded {len(data_mappings)} mapping dictionaries")

if data_mappings:
    print("\nAvailable mapping categories:")
    for key in data_mappings.keys():
        print(f"  - {key}: {len(data_mappings[key])} mappings")

# Apply mappings to training data
print("\n=== Applying Mappings to Training Data ===")
train_data_decoded = apply_data_mappings(train_data, data_mappings)
train_data_enhanced = create_meaningful_features(train_data_decoded)

# Apply mappings to test data
print("\n=== Applying Mappings to Test Data ===")
test_data_decoded = apply_data_mappings(test_data, data_mappings)
test_data_enhanced = create_meaningful_features(test_data_decoded)

print(f"\nTraining data enhanced shape: {train_data_enhanced.shape}")
print(f"Test data enhanced shape: {test_data_enhanced.shape}")

# Show sample of decoded columns
decoded_cols = [col for col in train_data_enhanced.columns if '_decoded' in col]
if decoded_cols:
    print(f"\nDecoded columns created: {len(decoded_cols)}")
    print("Sample decoded values:")
    for col in decoded_cols[:5]:  # Show first 5 decoded columns
        print(f"\n{col}:")
        print(train_data_enhanced[col].value_counts().head())

## 2. Data Dictionary Integration

Load and integrate the Hebrew data dictionaries to decode categorical variables in the accident dataset. This step is crucial for understanding the meaning behind the numerical codes used in the original data.

In [ ]:
# Examine the Excel dictionary structure in detail
import openpyxl

def examine_excel_structure():
    """
    Examine the structure of both Excel files to understand the data format
    """
    files_to_examine = [
        'data/dictionary22.xlsx',
        'data/data-dictionary-document-accidents22.xlsx'
    ]
    
    for file_path in files_to_examine:
        print(f"\n=== Examining {file_path.split('/')[-1]} ===")
        
        try:
            workbook = openpyxl.load_workbook(file_path)
            print(f"Sheets available: {workbook.sheetnames}")
            
            for sheet_name in workbook.sheetnames:
                sheet = workbook[sheet_name]
                print(f"\nSheet: {sheet_name}")
                print(f"Max row: {sheet.max_row}, Max column: {sheet.max_column}")
                
                # Print first few rows to understand structure
                print("First 5 rows of data:")
                for row in range(1, min(6, sheet.max_row + 1)):
                    row_data = []
                    for col in range(1, min(6, sheet.max_column + 1)):
                        cell_value = sheet.cell(row=row, column=col).value
                        row_data.append(str(cell_value) if cell_value is not None else "")
                    print(f"  Row {row}: {row_data}")
                
                if sheet.max_row > 5:
                    print("  ...")
            
            workbook.close()
            
        except Exception as e:
            print(f"Error examining {file_path}: {e}")

# Examine the structure
examine_excel_structure()

# Let's also check what columns we have in our data
print("\n=== Current Data Columns ===")
print("Training data columns:")
for i, col in enumerate(train_data.columns):
    print(f"  {i+1:2d}. {col}")

print(f"\nTotal columns: {len(train_data.columns)}")

# Check some sample values for key columns
print("\n=== Sample Values for Key Columns ===")
key_columns = ['MEZEG_AVIR', 'SUG_DEREH', 'MAHOZ', 'YOM_LAYLA', 'HUMRAT_TEUNA']
existing_key_cols = [col for col in key_columns if col in train_data.columns]

for col in existing_key_cols:
    print(f"\n{col}:")
    print(train_data[col].value_counts().head(10))

In [ ]:
# Create proper data mappings based on the Excel structure
def load_proper_mappings():
    """
    Load mappings from dictionary22.xlsx with proper structure understanding
    """
    mappings = {}
    
    try:
        workbook = openpyxl.load_workbook('data/dictionary22.xlsx')
        sheet = workbook['מילון']
        
        # Group by MS_TAVLA values
        current_category = None
        current_mappings = {}
        
        for row in range(2, sheet.max_row + 1):
            ms_tavla = sheet.cell(row=row, column=1).value
            kod = sheet.cell(row=row, column=2).value  
            teur = sheet.cell(row=row, column=3).value
            
            if ms_tavla == '0':  # Category header
                if current_category and current_mappings:
                    mappings[current_category] = current_mappings.copy()
                current_category = teur
                current_mappings = {}
            else:
                # This is a mapping entry
                if current_category and kod is not None and teur is not None:
                    try:
                        key = int(kod) if str(kod).isdigit() else kod
                        current_mappings[key] = str(teur).strip()
                    except:
                        pass
        
        # Don't forget the last category
        if current_category and current_mappings:
            mappings[current_category] = current_mappings.copy()
            
        workbook.close()
        
    except Exception as e:
        print(f"Error loading mappings: {e}")
    
    return mappings

def apply_proper_mappings(df):
    """
    Apply the loaded mappings to create decoded columns
    """
    df_decoded = df.copy()
    
    # Load mappings
    mappings = load_proper_mappings()
    
    # Define column to mapping relationships
    column_mappings = {
        'SUG_DEREH': 'סוג הדרך',
        'HUMRAT_TEUNA': 'חומרת התאונה', 
        'SUG_TEUNA': 'סוג התאונה',
        'YOM_LAYLA': 'יום/לילה',
        'MEZEG_AVIR': 'מזג אוויר',
        'MAHOZ': 'מחוז',
        'SUG_YOM': 'סוג יום',
        'TKINUT': 'תקינות הדרך',
        'PNE_KVISH': 'פני הכביש',
        'SIMUN_TIMRUR': 'סימון תימרור',
        'TEURA': 'תאורה',
        'MAAMAD_MINIZIPALI': 'מעמד מוניציפלי',
        'ZURAT_ISHUV': 'צורת יישוב'
    }
    
    print("=== Applying Proper Mappings ===")
    print(f"Available mapping categories: {list(mappings.keys())}")
    
    # Apply mappings
    for col, mapping_key in column_mappings.items():
        if col in df.columns and mapping_key in mappings:
            mapping_dict = mappings[mapping_key]
            df_decoded[f'{col}_decoded'] = df[col].map(mapping_dict)
            
            # Show mapping results
            mapped_count = df_decoded[f'{col}_decoded'].notna().sum()
            total_count = len(df_decoded)
            print(f"{col} -> {col}_decoded: {mapped_count}/{total_count} values mapped")
            
            # Show sample mappings
            print(f"  Sample mappings for {col}:")
            sample_original = df[col].value_counts().head(3)
            for val in sample_original.index:
                decoded_val = mapping_dict.get(val, 'Unknown')
                print(f"    {val} -> {decoded_val}")
            print()
    
    return df_decoded, mappings

def create_enhanced_features(df):
    """
    Create enhanced features based on decoded data
    """
    df_enhanced = df.copy()
    
    # Time-based features
    if 'SHAA' in df.columns:
        # Hour categories
        df_enhanced['hour_category'] = pd.cut(df['SHAA'], 
                                            bins=[0, 6, 12, 18, 24], 
                                            labels=['night', 'morning', 'afternoon', 'evening'],
                                            include_lowest=True)
        
        # Rush hour indicator (7-9 AM and 5-7 PM)
        df_enhanced['is_rush_hour'] = ((df['SHAA'] >= 7) & (df['SHAA'] <= 9)) | \
                                     ((df['SHAA'] >= 17) & (df['SHAA'] <= 19))
        
        # Peak traffic hours (broader definition)
        df_enhanced['is_peak_traffic'] = ((df['SHAA'] >= 6) & (df['SHAA'] <= 10)) | \
                                        ((df['SHAA'] >= 16) & (df['SHAA'] <= 20))
    
    # Weekend indicator
    if 'YOM_BASHAVUA' in df.columns:
        df_enhanced['is_weekend'] = df['YOM_BASHAVUA'].isin([6, 7])
    
    # Severity scoring
    if 'HUMRAT_TEUNA_decoded' in df.columns:
        severity_score_map = {
            'קלה': 1,
            'בינונית': 2, 
            'קשה': 3,
            'קטלנית': 4
        }
        df_enhanced['severity_score'] = df['HUMRAT_TEUNA_decoded'].map(severity_score_map)
    
    # Weather risk scoring  
    if 'MEZEG_AVIR_decoded' in df.columns:
        weather_risk_map = {
            'בהיר': 1,
            'מעונן': 2,
            'גשם קל': 3,
            'גשם': 4,
            'עב': 3,
            'ערפל': 4,
            'שלג': 5,
            'סערה': 5
        }
        df_enhanced['weather_risk_score'] = df['MEZEG_AVIR_decoded'].map(weather_risk_map)
    
    # Road type risk
    if 'SUG_DEREH_decoded' in df.columns:
        road_risk_map = {
            'דרך עירונית': 2,
            'דרך בין עירונית': 3,
            'כביש מהיר': 4,
            'אוטוסטרדה': 3
        }
        df_enhanced['road_risk_score'] = df['SUG_DEREH_decoded'].map(road_risk_map)
    
    return df_enhanced

# Apply proper mappings and feature enhancement
print("=== Loading and Applying Proper Data Mappings ===")

# Apply to training data
train_data_decoded, data_mappings = apply_proper_mappings(train_data)
train_data_final = create_enhanced_features(train_data_decoded)

print("\n" + "="*50)

# Apply to test data  
test_data_decoded, _ = apply_proper_mappings(test_data)
test_data_final = create_enhanced_features(test_data_decoded)

print(f"\nFinal training data shape: {train_data_final.shape}")
print(f"Final test data shape: {test_data_final.shape}")

# Show decoded columns created
decoded_cols = [col for col in train_data_final.columns if '_decoded' in col]
enhanced_cols = [col for col in train_data_final.columns if col not in train_data.columns and '_decoded' not in col]

print(f"\nDecoded columns created: {len(decoded_cols)}")
print(f"Enhanced feature columns created: {len(enhanced_cols)}")

print("\nSample of decoded data:")
for col in decoded_cols[:3]:
    print(f"\n{col}:")
    print(train_data_final[col].value_counts().head())

# Update the main dataframes for subsequent analysis
train_clean = train_data_final.copy()
test_clean = test_data_final.copy()

In [ ]:
# Create manual mappings based on Israeli traffic accident codes
def create_manual_mappings():
    """
    Create manual mappings for the most important columns
    """
    mappings = {
        # Road type mappings (SUG_DEREH)
        'SUG_DEREH': {
            1: 'inter_urban_road',
            2: 'urban_road', 
            3: 'access_road',
            4: 'highway'
        },
        
        # Accident severity (HUMRAT_TEUNA)
        'HUMRAT_TEUNA': {
            1: 'fatal',
            2: 'severe_injury',
            3: 'light_injury'
        },
        
        # Day/Night (YOM_LAYLA)
        'YOM_LAYLA': {
            1: 'day',
            5: 'night'
        },
        
        # Weather conditions (MEZEG_AVIR)
        'MEZEG_AVIR': {
            1: 'clear',
            2: 'cloudy',
            3: 'rain',
            4: 'fog',
            5: 'storm',
            9: 'unknown'
        },
        
        # Districts (MAHOZ) - based on Israeli districts
        'MAHOZ': {
            1: 'jerusalem',
            2: 'north',
            3: 'haifa', 
            4: 'center',
            5: 'tel_aviv',
            6: 'south',
            7: 'judea_samaria',
            9: 'unknown'
        },
        
        # Day type (SUG_YOM)
        'SUG_YOM': {
            1: 'weekday',
            2: 'friday',
            3: 'saturday',
            4: 'holiday_eve',
            5: 'holiday'
        }
    }
    
    return mappings

def apply_manual_mappings(df):
    """
    Apply manual mappings to the data
    """
    df_decoded = df.copy()
    mappings = create_manual_mappings()
    
    print("=== Applying Manual Mappings ===")
    
    for col, mapping_dict in mappings.items():
        if col in df.columns:
            df_decoded[f'{col}_decoded'] = df[col].map(mapping_dict)
            
            # Show mapping results
            mapped_count = df_decoded[f'{col}_decoded'].notna().sum()
            total_count = len(df_decoded)
            coverage = (mapped_count / total_count) * 100
            
            print(f"{col} -> {col}_decoded: {mapped_count}/{total_count} ({coverage:.1f}%) values mapped")
            
            # Show sample mappings
            print(f"  Sample mappings:")
            sample_original = df[col].value_counts().head(3)
            for val in sample_original.index:
                decoded_val = mapping_dict.get(val, 'unknown')
                count = sample_original[val]
                print(f"    {val} -> '{decoded_val}' ({count} cases)")
            print()
    
    return df_decoded

# Apply manual mappings
print("=== Applying Manual Mappings to Data ===")

# Apply to training data
train_data_decoded = apply_manual_mappings(train_data)
train_data_final = create_enhanced_features(train_data_decoded)

print("="*50)

# Apply to test data
test_data_decoded = apply_manual_mappings(test_data)  
test_data_final = create_enhanced_features(test_data_decoded)

print(f"\nFinal training data shape: {train_data_final.shape}")
print(f"Final test data shape: {test_data_final.shape}")

# Show created columns
decoded_cols = [col for col in train_data_final.columns if '_decoded' in col]
enhanced_cols = [col for col in train_data_final.columns if col not in train_data.columns and '_decoded' not in col]

print(f"\nDecoded columns created: {len(decoded_cols)}")
print(f"Enhanced feature columns created: {len(enhanced_cols)}")

print("\nDecoded columns:", decoded_cols)
print("Enhanced columns:", enhanced_cols)

# Update variables for further analysis
train_clean = train_data_final.copy()
test_clean = test_data_final.copy()

print("\n=== Sample Decoded Data ===")
sample_cols = ['HUMRAT_TEUNA', 'HUMRAT_TEUNA_decoded', 'SUG_DEREH', 'SUG_DEREH_decoded', 'YOM_LAYLA', 'YOM_LAYLA_decoded']
existing_sample_cols = [col for col in sample_cols if col in train_clean.columns]
print(train_clean[existing_sample_cols].head(10))

## 3. Data Cleaning and Preparation

Now that we have properly decoded the categorical variables using the data dictionary, let's examine the results. The decoded columns provide meaningful interpretations of the numerical codes, which will be essential for our analysis and modeling.

In [ ]:
# Updated data cleaning using the properly decoded data
def clean_decoded_data(df, is_train=True):
    """
    Clean and preprocess the decoded accident data.
    
    Args:
        df: DataFrame with decoded categorical variables
        is_train: Boolean indicating if this is training data
    
    Returns:
        pd.DataFrame: Cleaned dataframe
    """
    df_clean = df.copy()
    
    # Remove unnecessary columns
    columns_to_drop = [
        'pk_teuna_fikt',  # Fictitious accident number (ID)
        'sug_tik',        # File type
        'YEAR',           # Already have SHNAT_TEUNA
    ]
    
    existing_drops = [col for col in columns_to_drop if col in df_clean.columns]
    if existing_drops:
        df_clean = df_clean.drop(columns=existing_drops)
        print(f"Dropped columns: {existing_drops}")
    
    # Handle missing values in numeric columns
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    missing_numeric = df_clean[numeric_cols].isnull().sum()
    print(f"\nMissing values in numeric columns:")
    print(missing_numeric[missing_numeric > 0])
    
    # Fill missing numeric values with median
    for col in numeric_cols:
        if df_clean[col].isnull().sum() > 0:
            median_val = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(median_val)
    
    # Handle missing values in categorical columns (both original and decoded)
    categorical_cols = df_clean.select_dtypes(include=['object']).columns
    missing_categorical = df_clean[categorical_cols].isnull().sum()
    print(f"\nMissing values in categorical columns:")
    print(missing_categorical[missing_categorical > 0])
    
    # Fill missing categorical values with 'unknown'
    for col in categorical_cols:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col] = df_clean[col].fillna('unknown')
    
    # Data validation
    print(f"\n=== Data Validation ===")
    print(f"Original shape: {df.shape}")
    print(f"Cleaned shape: {df_clean.shape}")
    print(f"Total missing values after cleaning: {df_clean.isnull().sum().sum()}")
    
    # Validate key columns exist
    key_columns = ['HUMRAT_TEUNA', 'SHAA', 'YOM_LAYLA', 'SUG_DEREH', 'MEZEG_AVIR']
    missing_key_cols = [col for col in key_columns if col not in df_clean.columns]
    if missing_key_cols:
        print(f"Warning: Missing key columns: {missing_key_cols}")
    
    # Show distribution of target variable
    if 'HUMRAT_TEUNA' in df_clean.columns:
        print(f"\nTarget variable distribution:")
        target_dist = df_clean['HUMRAT_TEUNA'].value_counts().sort_index()
        print(target_dist)
        
        if 'HUMRAT_TEUNA_decoded' in df_clean.columns:
            print(f"\nTarget variable distribution (decoded):")
            target_dist_decoded = df_clean['HUMRAT_TEUNA_decoded'].value_counts()
            print(target_dist_decoded)
    
    return df_clean

# Since we already have properly decoded data, let's use it directly
print("=== Using Previously Decoded Data ===")
print(f"Training data with decoded features: {train_data_final.shape}")
print(f"Test data with decoded features: {test_data_final.shape}")

# Apply cleaning to the decoded data
train_clean = clean_decoded_data(train_data_final, is_train=True)
test_clean = clean_decoded_data(test_data_final, is_train=False)

print("\n=== Final Cleaned Data Summary ===")
print(f"Training data final shape: {train_clean.shape}")
print(f"Test data final shape: {test_clean.shape}")

## 4. Exploratory Data Analysis (EDA)

We explore the patterns in our data to understand accident severity distribution across different dimensions.

In [ ]:
# Show sample of important decoded columns
print("\n=== Sample of Key Decoded Features ===")
key_decoded_cols = [col for col in train_clean.columns if '_decoded' in col][:5]
key_enhanced_cols = ['hour_category', 'is_rush_hour', 'severity_score', 'weather_risk_score']
existing_enhanced = [col for col in key_enhanced_cols if col in train_clean.columns]

sample_display_cols = ['HUMRAT_TEUNA'] + key_decoded_cols + existing_enhanced
existing_display_cols = [col for col in sample_display_cols if col in train_clean.columns]

print(f"Displaying columns: {existing_display_cols}")
print(train_clean[existing_display_cols].head(10))

In [ ]:
# Enhanced EDA with decoded categorical variables
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Enhanced Exploratory Data Analysis with Decoded Variables', fontsize=16, fontweight='bold')

# 1. Accident Severity Distribution (Decoded)
if 'HUMRAT_TEUNA_decoded' in train_clean.columns:
    severity_counts = train_clean['HUMRAT_TEUNA_decoded'].value_counts()
    severity_pct = train_clean['HUMRAT_TEUNA_decoded'].value_counts(normalize=True) * 100
    
    bars = axes[0,0].bar(severity_counts.index, severity_counts.values, 
                        color=['red', 'orange', 'lightblue'])
    axes[0,0].set_title('Accident Severity Distribution')
    axes[0,0].set_xlabel('Severity Level')
    axes[0,0].set_ylabel('Number of Accidents')
    
    # Add percentage labels
    for i, (bar, pct) in enumerate(zip(bars, severity_pct.values)):
        height = bar.get_height()
        axes[0,0].text(bar.get_x() + bar.get_width()/2., height + 100,
                      f'{pct:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    axes[0,0].tick_params(axis='x', rotation=45)
else:
    # Fallback to original encoded data
    severity_counts = train_clean['HUMRAT_TEUNA'].value_counts().sort_index()
    axes[0,0].bar(severity_counts.index, severity_counts.values)
    axes[0,0].set_title('Accident Severity Distribution (Encoded)')

# 2. Road Type Distribution (Decoded)
if 'SUG_DEREH_decoded' in train_clean.columns:
    road_counts = train_clean['SUG_DEREH_decoded'].value_counts()
    axes[0,1].pie(road_counts.values, labels=road_counts.index, autopct='%1.1f%%', startangle=90)
    axes[0,1].set_title('Distribution by Road Type')
else:
    road_counts = train_clean['SUG_DEREH'].value_counts()
    axes[0,1].pie(road_counts.values, labels=[f'Type {i}' for i in road_counts.index], autopct='%1.1f%%')
    axes[0,1].set_title('Distribution by Road Type (Encoded)')

# 3. Day/Night Distribution (Decoded)
if 'YOM_LAYLA_decoded' in train_clean.columns:
    day_night_counts = train_clean['YOM_LAYLA_decoded'].value_counts()
    colors = ['gold', 'lightblue']
    axes[0,2].pie(day_night_counts.values, labels=day_night_counts.index, 
                 colors=colors, autopct='%1.1f%%', startangle=90)
    axes[0,2].set_title('Day vs Night Accidents')
else:
    day_night_counts = train_clean['YOM_LAYLA'].value_counts()
    axes[0,2].pie(day_night_counts.values, labels=[f'Code {i}' for i in day_night_counts.index], autopct='%1.1f%%')
    axes[0,2].set_title('Day vs Night Accidents (Encoded)')

# 4. Weather Conditions (Decoded)
if 'MEZEG_AVIR_decoded' in train_clean.columns:
    weather_counts = train_clean['MEZEG_AVIR_decoded'].value_counts()
    axes[1,0].barh(range(len(weather_counts)), weather_counts.values)
    axes[1,0].set_yticks(range(len(weather_counts)))
    axes[1,0].set_yticklabels(weather_counts.index)
    axes[1,0].set_title('Weather Conditions Distribution')
    axes[1,0].set_xlabel('Number of Accidents')
else:
    weather_counts = train_clean['MEZEG_AVIR'].value_counts()
    axes[1,0].bar(weather_counts.index, weather_counts.values)
    axes[1,0].set_title('Weather Conditions (Encoded)')

# 5. District Distribution (Decoded)
if 'MAHOZ_decoded' in train_clean.columns:
    district_counts = train_clean['MAHOZ_decoded'].value_counts()
    axes[1,1].barh(range(len(district_counts)), district_counts.values, 
                  color=plt.cm.Set3(range(len(district_counts))))
    axes[1,1].set_yticks(range(len(district_counts)))
    axes[1,1].set_yticklabels(district_counts.index)
    axes[1,1].set_title('Accidents by District')
    axes[1,1].set_xlabel('Number of Accidents')
else:
    district_counts = train_clean['MAHOZ'].value_counts()
    axes[1,1].bar(district_counts.index, district_counts.values)
    axes[1,1].set_title('Accidents by District (Encoded)')

# 6. Hour of Day Distribution
if 'SHAA' in train_clean.columns:
    # Create a complete range of hours (0-23) with counts
    all_hours = range(24)
    hourly_data = train_clean['SHAA'].value_counts()
    hourly_counts = [hourly_data.get(hour, 0) for hour in all_hours]
    
    axes[1,2].plot(all_hours, hourly_counts, marker='o', linewidth=2, markersize=4)
    axes[1,2].set_title('Accidents by Hour of Day')
    axes[1,2].set_xlabel('Hour (24-hour format)')
    axes[1,2].set_ylabel('Number of Accidents')
    axes[1,2].grid(True, alpha=0.3)
    
    # Set proper x-axis ticks and labels
    axes[1,2].set_xticks(range(0, 24, 2))
    axes[1,2].set_xticklabels([f'{h:02d}:00' for h in range(0, 24, 2)], rotation=45)
    axes[1,2].set_xlim(-0.5, 23.5)
    
    # Highlight peak hours
    hourly_series = pd.Series(hourly_counts, index=all_hours)
    peak_hours = hourly_series.nlargest(3)
    for hour, count in peak_hours.items():
        axes[1,2].scatter(hour, count, color='red', s=100, zorder=5)
        axes[1,2].annotate(f'{count}', (hour, count), xytext=(5, 5), 
                          textcoords='offset points', fontsize=8, color='red')

plt.tight_layout()
plt.show()

# Print key insights
print("=== Key EDA Insights with Decoded Data ===")

if 'HUMRAT_TEUNA_decoded' in train_clean.columns:
    print("\n1. Accident Severity:")
    severity_dist = train_clean['HUMRAT_TEUNA_decoded'].value_counts()
    for severity, count in severity_dist.items():
        pct = (count / len(train_clean)) * 100
        print(f"   {severity}: {count:,} ({pct:.1f}%)")

if 'SUG_DEREH_decoded' in train_clean.columns:
    print("\n2. Road Type Risk:")
    road_severity = pd.crosstab(train_clean['SUG_DEREH_decoded'], 
                               train_clean['HUMRAT_TEUNA_decoded'], 
                               normalize='index') * 100
    print(road_severity.round(1))

if 'YOM_LAYLA_decoded' in train_clean.columns:
    print("\n3. Day vs Night Risk:")
    day_night_severity = pd.crosstab(train_clean['YOM_LAYLA_decoded'], 
                                    train_clean['HUMRAT_TEUNA_decoded'], 
                                    normalize='index') * 100
    print(day_night_severity.round(1))

if 'MEZEG_AVIR_decoded' in train_clean.columns:
    print("\n4. Weather Impact:")
    weather_severity = pd.crosstab(train_clean['MEZEG_AVIR_decoded'], 
                                  train_clean['HUMRAT_TEUNA_decoded'], 
                                  normalize='index') * 100
    print(weather_severity.round(1))

# Enhanced features insights
if 'severity_score' in train_clean.columns:
    print(f"\n5. Average Severity Score: {train_clean['severity_score'].mean():.2f}")

if 'weather_risk_score' in train_clean.columns:
    print(f"6. Average Weather Risk Score: {train_clean['weather_risk_score'].mean():.2f}")

if 'is_rush_hour' in train_clean.columns:
    rush_hour_pct = (train_clean['is_rush_hour'].sum() / len(train_clean)) * 100
    print(f"7. Rush Hour Accidents: {rush_hour_pct:.1f}% of all accidents")

## 5. Feature Engineering

We create additional features to improve model performance and capture domain-specific patterns in road accident data.

In [ ]:
# Enhanced Feature Engineering with Decoded Data
def engineer_enhanced_features(df, is_train=True):
    """
    Engineer features using both original and decoded data for better model performance.
    
    Args:
        df: DataFrame with both original and decoded features
        is_train: Boolean indicating if this is training data
    
    Returns:
        pd.DataFrame: DataFrame with engineered features
    """
    df_features = df.copy()
    
    print(f"=== Enhanced Feature Engineering ({'Training' if is_train else 'Test'} Data) ===")
    
    # 1. Temporal Features
    if 'SHAA' in df_features.columns:
        # Cyclical encoding for hour
        df_features['hour_sin'] = np.sin(2 * np.pi * df_features['SHAA'] / 24)
        df_features['hour_cos'] = np.cos(2 * np.pi * df_features['SHAA'] / 24)
        
        # Rush hour indicators (multiple definitions)
        df_features['is_morning_rush'] = ((df_features['SHAA'] >= 7) & (df_features['SHAA'] <= 9)).astype(int)
        df_features['is_evening_rush'] = ((df_features['SHAA'] >= 17) & (df_features['SHAA'] <= 19)).astype(int)
        df_features['is_peak_traffic'] = ((df_features['SHAA'] >= 6) & (df_features['SHAA'] <= 10) | 
                                         (df_features['SHAA'] >= 16) & (df_features['SHAA'] <= 20)).astype(int)
        
        # Time categories
        df_features['time_period'] = pd.cut(df_features['SHAA'], 
                                          bins=[0, 6, 12, 18, 24], 
                                          labels=['night', 'morning', 'afternoon', 'evening'],
                                          include_lowest=True)
    
    # 2. Calendar Features
    if 'YOM_BASHAVUA' in df_features.columns:
        df_features['is_weekend'] = (df_features['YOM_BASHAVUA'].isin([6, 7])).astype(int)
        df_features['is_friday'] = (df_features['YOM_BASHAVUA'] == 6).astype(int)
        
    if 'HODESH_TEUNA' in df_features.columns:
        # Seasonal features
        df_features['season'] = pd.cut(df_features['HODESH_TEUNA'], 
                                     bins=[0, 3, 6, 9, 12], 
                                     labels=['winter', 'spring', 'summer', 'autumn'],
                                     include_lowest=True)
        
        # Summer indicator (high accident season)
        df_features['is_summer'] = (df_features['HODESH_TEUNA'].isin([6, 7, 8])).astype(int)
    
    # 3. Risk Scoring based on decoded data
    
    # Weather risk scoring
    if 'MEZEG_AVIR_decoded' in df_features.columns:
        weather_risk_mapping = {
            'clear': 1,
            'cloudy': 2,
            'rain': 4,
            'fog': 4,
            'storm': 5,
            'unknown': 3  # Medium risk for unknown
        }
        df_features['weather_risk_score'] = df_features['MEZEG_AVIR_decoded'].map(weather_risk_mapping).fillna(3)
    
    # Road type risk scoring
    if 'SUG_DEREH_decoded' in df_features.columns:
        road_risk_mapping = {
            'urban_road': 2,
            'inter_urban_road': 3,
            'access_road': 2,
            'highway': 4
        }
        df_features['road_risk_score'] = df_features['SUG_DEREH_decoded'].map(road_risk_mapping).fillna(2)
    
    # Day/Night risk factor
    if 'YOM_LAYLA_decoded' in df_features.columns:
        df_features['is_night'] = (df_features['YOM_LAYLA_decoded'] == 'night').astype(int)
    
    # 4. Geographic Features
    if 'X' in df_features.columns and 'Y' in df_features.columns:
        # Distance to major cities (using approximate coordinates)
        tel_aviv_x, tel_aviv_y = 181000, 664000  # Approximate Israeli grid coordinates
        jerusalem_x, jerusalem_y = 222000, 631000
        haifa_x, haifa_y = 171000, 744000
        
        df_features['dist_to_tel_aviv'] = np.sqrt((df_features['X'] - tel_aviv_x)**2 + 
                                                 (df_features['Y'] - tel_aviv_y)**2)
        df_features['dist_to_jerusalem'] = np.sqrt((df_features['X'] - jerusalem_x)**2 + 
                                                  (df_features['Y'] - jerusalem_y)**2)
        df_features['dist_to_haifa'] = np.sqrt((df_features['X'] - haifa_x)**2 + 
                                              (df_features['Y'] - haifa_y)**2)
        
        # Closest major city
        distances = df_features[['dist_to_tel_aviv', 'dist_to_jerusalem', 'dist_to_haifa']]
        df_features['closest_city'] = distances.idxmin(axis=1).str.replace('dist_to_', '')
        
        # Urban proximity (closer to major cities = more urban)
        df_features['min_city_distance'] = distances.min(axis=1)
        df_features['is_urban_area'] = (df_features['min_city_distance'] < 20000).astype(int)  # within 20km
    
    # 5. Interaction Features
    
    # Critical combinations
    if 'is_night' in df_features.columns and 'weather_risk_score' in df_features.columns:
        df_features['night_weather_risk'] = df_features['is_night'] * df_features['weather_risk_score']
    
    if 'road_risk_score' in df_features.columns and 'weather_risk_score' in df_features.columns:
        df_features['road_weather_risk'] = df_features['road_risk_score'] * df_features['weather_risk_score']
    
    if 'is_weekend' in df_features.columns and 'is_night' in df_features.columns:
        df_features['weekend_night'] = df_features['is_weekend'] * df_features['is_night']
    
    # High risk scenario
    if all(col in df_features.columns for col in ['is_night', 'weather_risk_score', 'road_risk_score']):
        df_features['high_risk_scenario'] = (
            (df_features['is_night'] == 1) & 
            (df_features['weather_risk_score'] >= 4) & 
            (df_features['road_risk_score'] >= 3)
        ).astype(int)
    
    # 6. Speed and Infrastructure Features
    if 'MEHIRUT_MUTERET' in df_features.columns:
        # Speed categories
        df_features['speed_category'] = pd.cut(df_features['MEHIRUT_MUTERET'], 
                                             bins=[0, 50, 80, 120, 150], 
                                             labels=['low', 'medium', 'high', 'very_high'],
                                             include_lowest=True)
        
        # High speed indicator
        df_features['is_high_speed'] = (df_features['MEHIRUT_MUTERET'] >= 80).astype(int)
    
    # 7. One-hot encode important categorical variables
    categorical_features = []
    
    # Encode decoded categorical variables
    if 'MAHOZ_decoded' in df_features.columns:
        district_dummies = pd.get_dummies(df_features['MAHOZ_decoded'], prefix='district')
        df_features = pd.concat([df_features, district_dummies], axis=1)
        categorical_features.append('MAHOZ_decoded')
    
    if 'time_period' in df_features.columns:
        time_dummies = pd.get_dummies(df_features['time_period'], prefix='time')
        df_features = pd.concat([df_features, time_dummies], axis=1)
        categorical_features.append('time_period')
    
    if 'season' in df_features.columns:
        season_dummies = pd.get_dummies(df_features['season'], prefix='season')
        df_features = pd.concat([df_features, season_dummies], axis=1)
        categorical_features.append('season')
    
    if 'closest_city' in df_features.columns:
        city_dummies = pd.get_dummies(df_features['closest_city'], prefix='closest')
        df_features = pd.concat([df_features, city_dummies], axis=1)
        categorical_features.append('closest_city')
    
    # Remove original categorical columns that have been encoded
    df_features = df_features.drop(columns=categorical_features, errors='ignore')
    
    print(f"Features engineered successfully!")
    print(f"Original columns: {df.shape[1]}")
    print(f"Final columns: {df_features.shape[1]}")
    print(f"New features added: {df_features.shape[1] - df.shape[1]}")
    
    return df_features

# Apply enhanced feature engineering
train_features = engineer_enhanced_features(train_clean, is_train=True)
test_features = engineer_enhanced_features(test_clean, is_train=False)

print("\n=== Feature Engineering Complete ===")
print(f"Training features shape: {train_features.shape}")
print(f"Test features shape: {test_features.shape}")

# Show some of the new features created
new_feature_examples = [
    'hour_sin', 'hour_cos', 'weather_risk_score', 'road_risk_score',
    'is_night', 'is_weekend', 'dist_to_tel_aviv', 'night_weather_risk'
]
existing_new_features = [col for col in new_feature_examples if col in train_features.columns]

print(f"\nSample of new engineered features:")
print(f"Features: {existing_new_features}")
if existing_new_features:
    print(train_features[existing_new_features].head())

# Check feature consistency between train and test
train_cols = set(train_features.columns)
test_cols = set(test_features.columns)
missing_in_test = train_cols - test_cols
missing_in_train = test_cols - train_cols

if missing_in_test:
    print(f"\nWarning: Features missing in test set: {missing_in_test}")
if missing_in_train:
    print(f"Warning: Features missing in training set: {missing_in_train}")

# Align columns between train and test
common_features = list(train_cols & test_cols)
if 'HUMRAT_TEUNA' in common_features:
    common_features.remove('HUMRAT_TEUNA')  # Remove target variable

train_features_final = train_features[common_features + ['HUMRAT_TEUNA']]
test_features_final = test_features[common_features + ['HUMRAT_TEUNA']]

print(f"\nFinal aligned features: {len(common_features)}")
print(f"Train features final shape: {train_features_final.shape}")
print(f"Test features final shape: {test_features_final.shape}")

## 6. Modeling Pipeline

We train multiple machine learning models to predict accident severity using our engineered features.

In [ ]:
# Model Training with Enhanced Features - Fixed NaN handling
def prepare_model_data(train_df, test_df, target_col='HUMRAT_TEUNA'):
    """
    Prepare data for model training with proper feature selection and encoding.
    
    Args:
        train_df: Training dataframe with engineered features
        test_df: Test dataframe with engineered features  
        target_col: Name of target column
        
    Returns:
        tuple: (X_train, X_test, y_train, y_test, feature_names)
    """
    print("=== Preparing Model Data ===")
    
    # Select features for modeling (exclude target and ID columns)
    exclude_cols = [
        target_col,
        'pk_teuna_fikt', 'sug_tik',  # ID columns
        # Exclude original categorical columns if decoded versions exist
        'SUG_DEREH', 'MEZEG_AVIR', 'MAHOZ', 'YOM_LAYLA', 'SUG_YOM',  
        # Exclude text/decoded categorical columns (already one-hot encoded)
        'SUG_DEREH_decoded', 'MEZEG_AVIR_decoded', 'MAHOZ_decoded', 
        'YOM_LAYLA_decoded', 'SUG_YOM_decoded'
    ]
    
    # Get feature columns
    feature_cols = [col for col in train_df.columns if col not in exclude_cols]
    
    # Ensure only numeric features are used
    numeric_features = []
    for col in feature_cols:
        if col in train_df.columns:
            if train_df[col].dtype in ['int64', 'float64', 'int32', 'float32', 'bool']:
                numeric_features.append(col)
            elif train_df[col].dtype == 'object':
                try:
                    # Try to convert to numeric
                    pd.to_numeric(train_df[col])
                    numeric_features.append(col)
                except:
                    print(f"Skipping non-numeric column: {col}")
    
    # Prepare training data
    X_train = train_df[numeric_features].copy()
    y_train = train_df[target_col].copy()
    
    # Prepare test data
    X_test = test_df[numeric_features].copy()
    y_test = test_df[target_col].copy()
    
    # Check for NaN values before imputation
    print(f"NaN values in training data before cleaning:")
    nan_counts = X_train.isnull().sum()
    nan_columns = nan_counts[nan_counts > 0]
    if len(nan_columns) > 0:
        print(nan_columns)
    else:
        print("No NaN values found")
    
    # Handle any remaining missing values with more robust imputation
    # For numeric columns, use median
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if X_train[col].isnull().any():
            median_val = X_train[col].median()
            if pd.isna(median_val):  # If median is also NaN, use 0
                median_val = 0
            X_train[col] = X_train[col].fillna(median_val)
            X_test[col] = X_test[col].fillna(median_val)
    
    # For any categorical columns that slipped through, use mode or 0
    for col in X_train.columns:
        if X_train[col].isnull().any():
            if X_train[col].dtype == 'object':
                mode_val = X_train[col].mode()
                fill_val = mode_val[0] if len(mode_val) > 0 else 'unknown'
            else:
                fill_val = 0
            X_train[col] = X_train[col].fillna(fill_val)
            X_test[col] = X_test[col].fillna(fill_val)
    
    # Convert any remaining object types to numeric if possible
    for col in X_train.columns:
        if X_train[col].dtype == 'object':
            try:
                X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
                X_test[col] = pd.to_numeric(X_test[col], errors='coerce')
                # Fill any new NaNs created by coercion
                X_train[col] = X_train[col].fillna(0)
                X_test[col] = X_test[col].fillna(0)
            except:
                # If conversion fails, remove the column
                print(f"Removing non-convertible column: {col}")
                X_train = X_train.drop(columns=[col])
                X_test = X_test.drop(columns=[col])
                numeric_features.remove(col)
    
    # Final check for NaN or infinite values
    X_train = X_train.replace([np.inf, -np.inf], 0)
    X_test = X_test.replace([np.inf, -np.inf], 0)
    
    # Final NaN check
    final_nans_train = X_train.isnull().sum().sum()
    final_nans_test = X_test.isnull().sum().sum()
    
    if final_nans_train > 0 or final_nans_test > 0:
        print(f"Warning: Still have {final_nans_train} NaNs in train and {final_nans_test} NaNs in test")
        X_train = X_train.fillna(0)
        X_test = X_test.fillna(0)
    
    print(f"Training samples: {X_train.shape[0]}")
    print(f"Test samples: {X_test.shape[0]}")
    print(f"Number of features: {X_train.shape[1]}")
    print(f"Target distribution (training):")
    print(y_train.value_counts().sort_index())
    
    return X_train, X_test, y_train, y_test, list(X_train.columns)

# Prepare data for modeling
X_train, X_test, y_train, y_test, feature_names = prepare_model_data(
    train_features_final, test_features_final
)

# Define models with balanced class weights for imbalanced data
models = {
    'Decision Tree': DecisionTreeClassifier(
        random_state=42,
        class_weight='balanced',
        max_depth=15,
        min_samples_split=50,
        min_samples_leaf=20
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced',
        max_depth=15,
        min_samples_split=50,
        min_samples_leaf=20,
        n_jobs=-1
    ),
    'AdaBoost': AdaBoostClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    )
}

# Train models and evaluate with cross-validation
print("\n=== Training Models with Enhanced Features ===")
trained_models = {}
cv_scores = {}

# Use stratified k-fold for better evaluation with imbalanced classes
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train the model
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='accuracy')
    cv_scores[name] = scores
    
    print(f"{name} - CV Accuracy: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")
    
    # Additional scoring metrics
    cv_scores_weighted = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1_weighted')
    print(f"{name} - CV F1-Weighted: {cv_scores_weighted.mean():.4f} (+/- {cv_scores_weighted.std() * 2:.4f})")

print("\n=== Cross-Validation Results Summary ===")
results = {}
for name in models.keys():
    results[name] = {
        'mean_cv_accuracy': cv_scores[name].mean(),
        'std_cv_accuracy': cv_scores[name].std(),
        'model': trained_models[name]
    }

# Sort by CV accuracy
sorted_results = sorted(results.items(), key=lambda x: x[1]['mean_cv_accuracy'], reverse=True)

print("Model ranking by CV accuracy:")
for i, (name, metrics) in enumerate(sorted_results, 1):
    print(f"{i}. {name}: {metrics['mean_cv_accuracy']:.4f} (+/- {metrics['std_cv_accuracy']*2:.4f})")

# Select best model
best_model_name = sorted_results[0][0]
best_model = sorted_results[0][1]['model']

print(f"\nBest model: {best_model_name}")
print(f"Best CV accuracy: {sorted_results[0][1]['mean_cv_accuracy']:.4f}")

# Feature importance for the best model
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(f"\nTop 10 most important features in {best_model_name}:")
    for idx, row in feature_importance.head(10).iterrows():
        print(f"  {row['feature']}: {row['importance']:.4f}")

print(f"\n=== Training Complete ===")
print(f"Models trained: {len(trained_models)}")
print(f"Features used: {len(feature_names)}")
print(f"Best performing model: {best_model_name}")

## 7. Evaluation on 2022 Test Data

We evaluate our trained models on the 2022 test data to assess temporal generalization performance.

In [ ]:
# Model Evaluation on Test Data (Temporal Generalization)
print("=== Evaluating Models on 2022 Test Data (Temporal Generalization) ===")

if 'trained_models' in locals() and len(trained_models) > 0:
    evaluation_results = {}
    
    for name, model in trained_models.items():
        print(f"\n--- Evaluating {name} ---")
        
        # Predictions on test set
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test) if hasattr(model, 'predict_proba') else None
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        
        evaluation_results[name] = {
            'accuracy': accuracy,
            'predictions': y_pred,
            'probabilities': y_pred_proba
        }
        
        print(f"Test Accuracy: {accuracy:.4f}")
        
        # Classification report
        print("\nClassification Report:")
        class_names = ['Fatal', 'Severe Injury', 'Light Injury']
        print(classification_report(y_test, y_pred, target_names=class_names))
        
        # Confusion Matrix
        cm = confusion_matrix(y_test, y_pred)
        print(f"\nConfusion Matrix:")
        print(cm)
        
        # Class-wise performance
        print(f"\nClass-wise Performance:")
        for i, class_name in enumerate(class_names):
            class_mask = (y_test == i+1)
            if class_mask.sum() > 0:
                class_accuracy = accuracy_score(y_test[class_mask], y_pred[class_mask])
                print(f"  {class_name}: {class_accuracy:.4f} ({class_mask.sum()} samples)")
    
    # Compare models
    print(f"\n=== Model Comparison on Test Data ===")
    print("Model Performance Summary:")
    sorted_eval = sorted(evaluation_results.items(), key=lambda x: x[1]['accuracy'], reverse=True)
    
    for i, (name, results) in enumerate(sorted_eval, 1):
        print(f"{i}. {name}: {results['accuracy']:.4f}")
    
    # Best model for temporal generalization
    best_test_model = sorted_eval[0][0]
    best_test_accuracy = sorted_eval[0][1]['accuracy']
    
    print(f"\nBest model on test data: {best_test_model}")
    print(f"Best test accuracy: {best_test_accuracy:.4f}")
    
    # Temporal generalization analysis
    if 'results' in locals() and best_model_name in results:
        cv_accuracy = results[best_model_name]['mean_cv_accuracy']
        test_accuracy = evaluation_results[best_model_name]['accuracy']
        generalization_gap = cv_accuracy - test_accuracy
        
        print(f"\n=== Temporal Generalization Analysis ===")
        print(f"Cross-validation accuracy (2020-2021): {cv_accuracy:.4f}")
        print(f"Test accuracy (2022): {test_accuracy:.4f}")
        print(f"Generalization gap: {generalization_gap:.4f}")
        
        if generalization_gap > 0.05:
            print("⚠️  Significant temporal drift detected!")
        elif generalization_gap > 0.02:
            print("⚠️  Moderate temporal drift detected.")
        else:
            print("✅ Good temporal generalization.")
    elif 'cv_scores' in locals():
        # Alternative approach using cv_scores
        cv_accuracy = cv_scores[best_model_name].mean()
        test_accuracy = evaluation_results[best_model_name]['accuracy']
        generalization_gap = cv_accuracy - test_accuracy
        
        print(f"\n=== Temporal Generalization Analysis ===")
        print(f"Cross-validation accuracy (2020-2021): {cv_accuracy:.4f}")
        print(f"Test accuracy (2022): {test_accuracy:.4f}")
        print(f"Generalization gap: {generalization_gap:.4f}")
        
        if generalization_gap > 0.05:
            print("⚠️  Significant temporal drift detected!")
        elif generalization_gap > 0.02:
            print("⚠️  Moderate temporal drift detected.")
        else:
            print("✅ Good temporal generalization.")
    
    # Plot confusion matrix for best model
    plt.figure(figsize=(12, 4))
    
    # Plot for best CV model
    plt.subplot(1, 2, 1)
    best_cv_model = trained_models[best_model_name]
    y_pred_cv = best_cv_model.predict(X_test)
    cm_cv = confusion_matrix(y_test, y_pred_cv)
    
    sns.heatmap(cm_cv, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Fatal', 'Severe', 'Light'],
                yticklabels=['Fatal', 'Severe', 'Light'])
    plt.title(f'Confusion Matrix - {best_model_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    
    # Plot for best test model if different
    if best_test_model != best_model_name:
        plt.subplot(1, 2, 2)
        best_test_model_obj = trained_models[best_test_model]
        y_pred_test = best_test_model_obj.predict(X_test)
        cm_test = confusion_matrix(y_test, y_pred_test)
        
        sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens',
                    xticklabels=['Fatal', 'Severe', 'Light'],
                    yticklabels=['Fatal', 'Severe', 'Light'])
        plt.title(f'Confusion Matrix - {best_test_model}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
    
    plt.tight_layout()
    plt.show()
    
    # Class distribution comparison
    print(f"\n=== Target Distribution Comparison ===")
    print("Training data (2020-2021):")
    train_dist = y_train.value_counts().sort_index()
    train_pct = (train_dist / len(y_train) * 100).round(1)
    for i, (count, pct) in enumerate(zip(train_dist.values, train_pct.values), 1):
        severity = ['Fatal', 'Severe Injury', 'Light Injury'][i-1]
        print(f"  {severity}: {count} ({pct}%)")
    
    print("\nTest data (2022):")
    test_dist = y_test.value_counts().sort_index()
    test_pct = (test_dist / len(y_test) * 100).round(1)
    for i, (count, pct) in enumerate(zip(test_dist.values, test_pct.values), 1):
        severity = ['Fatal', 'Severe Injury', 'Light Injury'][i-1]
        print(f"  {severity}: {count} ({pct}%)")
    
    # Distribution shift analysis
    print("\nDistribution shift:")
    for i in range(3):
        shift = test_pct.iloc[i] - train_pct.iloc[i]
        severity = ['Fatal', 'Severe Injury', 'Light Injury'][i]
        print(f"  {severity}: {shift:+.1f}% change")

else:
    print("Cannot evaluate models - no trained models available")
    print("Please run the model training cell first.")

In [ ]:
# Train baseline and ensemble models
def train_models(X_train, y_train):
    """
    Train multiple machine learning models for accident severity prediction.
    
    Args:
        X_train: Training features
        y_train: Training target
    
    Returns:
        dict: Dictionary of trained models
    """
    print("=== Training Models ===")
    
    models = {}
    
    # 1. Baseline Decision Tree
    print("Training Decision Tree (Baseline)...")
    dt_baseline = DecisionTreeClassifier(
        random_state=42,
        max_depth=10,
        min_samples_split=100,
        min_samples_leaf=50,
        class_weight='balanced'  # Handle class imbalance
    )
    dt_baseline.fit(X_train, y_train)
    models['Decision Tree'] = dt_baseline
    
    # 2. Random Forest with class balancing
    print("Training Random Forest...")
    rf_model = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        max_depth=15,
        min_samples_split=50,
        min_samples_leaf=20,
        class_weight='balanced',
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)
    models['Random Forest'] = rf_model
    
    # 3. AdaBoost with class balancing
    print("Training AdaBoost...")
    ada_model = AdaBoostClassifier(
        n_estimators=100,
        random_state=42,
        learning_rate=0.1,
        algorithm='SAMME'  # Better for multiclass
    )
    ada_model.fit(X_train, y_train)
    models['AdaBoost'] = ada_model
    
    print(f"Successfully trained {len(models)} models")
    return models

# Train all models
if X_train is not None and y_train is not None:
    trained_models = train_models(X_train, y_train)
    
    # Quick cross-validation to compare models
    print("\n=== Cross-Validation Scores (5-fold) ===")
    cv_scores = {}
    
    for name, model in trained_models.items():
        scores = cross_val_score(
            model, X_train, y_train, 
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
            scoring='f1_macro'
        )
        cv_scores[name] = scores
        print(f"{name}: {scores.mean():.3f} (+/- {scores.std() * 2:.3f})")
    
else:
    print("Cannot train models - data preparation failed")

In [ ]:
# Hyperparameter tuning for the best model
def tune_best_model(X_train, y_train, cv_scores):
    """
    Perform hyperparameter tuning on the best performing model.
    
    Args:
        X_train: Training features
        y_train: Training target  
        cv_scores: Cross-validation scores from previous step
    
    Returns:
        Best tuned model
    """
    print("=== Hyperparameter Tuning ===")
    
    # Find best model based on CV scores
    best_model_name = max(cv_scores.keys(), key=lambda k: cv_scores[k].mean())
    print(f"Best model for tuning: {best_model_name}")
    
    if best_model_name == 'Random Forest':
        # Random Forest hyperparameter grid (reduced for faster execution)
        param_grid = {
            'n_estimators': [50, 100],
            'max_depth': [10, 15],
            'min_samples_split': [50, 100],
            'min_samples_leaf': [20, 50]
        }
        base_model = RandomForestClassifier(
            random_state=42, 
            class_weight='balanced',
            n_jobs=-1
        )
        
    elif best_model_name == 'AdaBoost':
        # AdaBoost hyperparameter grid (reduced for faster execution)
        param_grid = {
            'n_estimators': [50, 100],
            'learning_rate': [0.1, 0.5],
            'algorithm': ['SAMME']
        }
        base_model = AdaBoostClassifier(random_state=42)
        
    else:  # Decision Tree
        # Decision Tree hyperparameter grid (reduced for faster execution)
        param_grid = {
            'max_depth': [10, 15],
            'min_samples_split': [50, 100],
            'min_samples_leaf': [20, 50],
            'criterion': ['gini', 'entropy']
        }
        base_model = DecisionTreeClassifier(
            random_state=42, 
            class_weight='balanced'
        )
    
    # Perform grid search with reduced CV folds for faster execution
    print(f"Tuning {best_model_name} with {len(param_grid)} parameters...")
    print(f"Total combinations: {np.prod([len(v) for v in param_grid.values()])}")
    
    grid_search = GridSearchCV(
        base_model,
        param_grid,
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),  # Reduced from 5 to 3 folds
        scoring='f1_macro',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best CV score: {grid_search.best_score_:.3f}")
    
    return grid_search.best_estimator_, best_model_name

# Perform hyperparameter tuning if models were trained successfully
if 'trained_models' in locals() and cv_scores:
    best_tuned_model, best_model_name = tune_best_model(X_train, y_train, cv_scores)
    print(f"\nBest tuned model: {best_model_name}")
else:
    print("Skipping hyperparameter tuning - no trained models available")

## 8. Feature Importance Analysis

Understanding which features drive accident severity predictions helps us interpret our models and provides insights for road safety interventions.

In [ ]:
# Comprehensive evaluation on test data
def evaluate_models(models, X_test, y_test):
    """
    Evaluate all models on test data and provide detailed metrics.
    
    Args:
        models (dict): Dictionary of trained models
        X_test: Test features
        y_test: Test target
    
    Returns:
        dict: Evaluation results for each model
    """
    print("=== Evaluating Models on 2022 Test Data ===")
    
    results = {}
    severity_labels = ['Fatal (1)', 'Severe (2)', 'Minor (3)']
    
    for name, model in models.items():
        print(f"\n--- {name} ---")
        
        # Make predictions
        y_pred = model.predict(X_test)
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        report = classification_report(y_test, y_pred, output_dict=True)
        
        # Store results
        results[name] = {
            'accuracy': accuracy,
            'predictions': y_pred,
            'classification_report': report
        }
        
        print(f"Accuracy: {accuracy:.3f}")
        print(f"Macro F1: {report['macro avg']['f1-score']:.3f}")
        print(f"Fatal Recall: {report.get('1', {}).get('recall', 0):.3f}")
        print(f"Severe Recall: {report.get('2', {}).get('recall', 0):.3f}")
        print(f"Minor Recall: {report.get('3', {}).get('recall', 0):.3f}")
    
    return results

# Evaluate all models if test data is available
if X_test is not None and y_test is not None and 'trained_models' in locals():
    # Define severity labels for confusion matrix
    severity_labels = ['Fatal (1)', 'Severe (2)', 'Minor (3)']
    
    evaluation_results = evaluate_models(trained_models, X_test, y_test)
    
    # Add best tuned model to evaluation if available
    if 'best_tuned_model' in locals():
        tuned_results = evaluate_models({f'Tuned {best_model_name}': best_tuned_model}, X_test, y_test)
        evaluation_results.update(tuned_results)
    
    # Create confusion matrices
    print("\n=== Confusion Matrices ===")
    
    n_models = len(evaluation_results)
    fig, axes = plt.subplots(1, n_models, figsize=(5*n_models, 4))
    if n_models == 1:
        axes = [axes]
    
    for idx, (name, results) in enumerate(evaluation_results.items()):
        cm = confusion_matrix(y_test, results['predictions'])
        
        sns.heatmap(cm, 
                   annot=True, 
                   fmt='d', 
                   cmap='Blues',
                   xticklabels=severity_labels,
                   yticklabels=severity_labels,
                   ax=axes[idx])
        axes[idx].set_title(f'{name}\nAccuracy: {results["accuracy"]:.3f}')
        axes[idx].set_xlabel('Predicted')
        axes[idx].set_ylabel('Actual')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("Cannot evaluate models - missing test data or trained models")

### Temporal Generalization Analysis

Our models were trained on 2020-2021 data and tested on 2022 data. This temporal split simulates real-world deployment scenarios where we need to predict future accidents based on historical patterns.

**Key Observations:**

1. **Generalization Gap:** We observe how well our models generalize to unseen temporal data. Any significant drop in performance between cross-validation scores (2020-2021) and test scores (2022) indicates temporal drift.

2. **Class Imbalance Impact:** Fatal accidents are rare but critical to predict correctly. We emphasize recall for fatal accidents as missing a fatal accident prediction has severe consequences.

3. **Feature Stability:** The importance of features that remain stable over time vs. those that may change due to external factors (policy changes, infrastructure improvements, etc.).

## 8. Feature Importance Analysis (Continued)

Understanding which features drive accident severity predictions helps us interpret our models and provides insights for road safety interventions.

In [ ]:
# Feature importance analysis
def analyze_feature_importance(model, feature_names, model_name, top_k=20):
    """
    Analyze and visualize feature importance for tree-based models.
    
    Args:
        model: Trained model with feature_importances_ attribute
        feature_names: List of feature names
        model_name: Name of the model for plotting
        top_k: Number of top features to display
    
    Returns:
        pd.DataFrame: Feature importance dataframe
    """
    if not hasattr(model, 'feature_importances_'):
        print(f"{model_name} doesn't have feature_importances_ attribute")
        return None
    
    # Create feature importance dataframe
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Plot top-k features
    plt.figure(figsize=(12, 8))
    top_features = importance_df.head(top_k)
    
    bars = plt.barh(range(len(top_features)), top_features['importance'])
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Feature Importance')
    plt.title(f'Top {top_k} Feature Importances - {model_name}')
    plt.gca().invert_yaxis()
    
    # Add value labels on bars
    for i, (idx, row) in enumerate(top_features.iterrows()):
        plt.text(row['importance'], i, f'{row["importance"]:.3f}', 
                va='center', ha='left', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    return importance_df

# Analyze feature importance for tree-based models
if 'trained_models' in locals() and X_train is not None:
    feature_names = X_train.columns.tolist()
    
    for name, model in trained_models.items():
        if hasattr(model, 'feature_importances_'):
            print(f"\n=== Feature Importance: {name} ===")
            importance_df = analyze_feature_importance(model, feature_names, name)
            
            if importance_df is not None:
                print("Top 10 most important features:")
                for idx, row in importance_df.head(10).iterrows():
                    print(f"{row['feature']}: {row['importance']:.4f}")
    
    # Analyze best tuned model if available
    if 'best_tuned_model' in locals():
        print(f"\n=== Feature Importance: Tuned {best_model_name} ===")
        tuned_importance = analyze_feature_importance(
            best_tuned_model, feature_names, f'Tuned {best_model_name}'
        )
        
        if tuned_importance is not None:
            print("Top 10 most important features (tuned model):")
            for idx, row in tuned_importance.head(10).iterrows():
                print(f"{row['feature']}: {row['importance']:.4f}")

else:
    print("Cannot analyze feature importance - no trained models available")

### Feature Importance Interpretation

The feature importance analysis reveals key factors that influence accident severity:

**Spatial Factors:**
- **Location coordinates (X, Y)** and **distance to major cities** often rank highly, indicating that geographic location significantly impacts accident severity
- Urban vs. rural areas may have different accident patterns

**Temporal Factors:**
- **Hour of day** (both raw and cyclical encoding) captures the impact of traffic patterns, visibility, and driver alertness
- **Day of week** reflects different traffic patterns between weekdays and weekends

**Infrastructure Factors:**
- **Road type** and **speed limits** directly relate to accident severity - highways typically have more severe accidents
- **Light conditions** (day/night) affect visibility and reaction times

**Environmental Factors:**
- **Weather conditions** impact road conditions and visibility
- **Interaction features** (e.g., road type × weather condition) capture complex relationships

These insights guide our safety recommendations and help prioritize intervention strategies.

## 9. Conclusions & Road Safety Recommendations

Based on our temporal generalization study of road accident severity prediction, we provide actionable recommendations for improving road safety in Israel.

## 10. Next Steps & Future Work

**Data Enhancements:**
- Incorporate additional external data sources (traffic volume, road quality indices)
- Include weather API data for more granular weather conditions
- Add demographic and socioeconomic factors by geographic region

**Model Improvements:**
- Experiment with ensemble methods and deep learning approaches
- Implement time series forecasting for seasonal accident prediction
- Develop separate models for different accident types and severity levels

**Deployment Considerations:**
- Create real-time prediction system for traffic management centers
- Develop mobile app for drivers with risk warnings
- Build dashboard for policy makers with actionable insights

**Policy Applications:**
- Use model insights to inform infrastructure investment decisions
- Guide targeted safety campaigns based on high-risk patterns
- Support evidence-based policy making for traffic safety regulations